In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("insurance.csv")

In [ ]:
df

# EDA

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
numeric_columns = ['age','bmi','children','charges']
for col in numeric_columns:
    plt.figure(figsize=(12,6))
    sns.histplot(df[col],kde = True, bins=20)


In [ ]:
sns.countplot(x= df['children'])

In [ ]:
sns.countplot(x='sex',data=df)

In [ ]:
sns.countplot(x = df['smoker'])

In [ ]:
for col in numeric_columns:
    plt.figure(figsize=(12,6))
    sns.boxplot(x=df[col])

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True),annot=True)

# Date cleaning and processing

In [ ]:
df_cleaned = df.copy()

In [ ]:
df_cleaned.drop_duplicates(inplace=True)

In [ ]:
df_cleaned.isnull().sum()

In [ ]:
df_cleaned.dtypes

In [ ]:
df_cleaned['sex'].value_counts()

In [ ]:
df_cleaned['sex'] = df_cleaned['sex'].map({
    "male":0,
    "female":1
})

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned['smoker'].value_counts()

In [ ]:
df_cleaned['smoker'] = df_cleaned['smoker'].map({
    "yes":1,
    "no":0
})

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.rename(columns={
    "sex" : "is_female", 
    "smoker" : "is_smoker"
},inplace=True)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned['region'].value_counts()

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned,columns=['region'],drop_first=True)

In [ ]:
df_cleaned

In [ ]:
df_cleaned =df_cleaned.astype(int)

In [ ]:
df_cleaned

# Feature Engineering and Extraction

In [ ]:
sns.histplot(df_cleaned['bmi'])

In [ ]:
df_cleaned['bmi_category'] = pd.cut(
    df_cleaned['bmi'],
    bins = [0, 18.5, 25, 30, float('inf')],
    labels = ['underwight','normal','overweight','obese']
)

In [ ]:
df_cleaned

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned,columns=['bmi_category'],drop_first=True)

In [ ]:
df_cleaned = df_cleaned.astype(int)

In [ ]:
df_cleaned

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
cols = ['age','bmi','children']
df_cleaned[cols] = scaler.fit_transform(df_cleaned[cols])



In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.info()

In [ ]:
df_cleaned.columns

In [ ]:
from scipy.stats import pearsonr

In [ ]:
selected_features = ['age', 'bmi', 'children', 'is_female', 'is_smoker',
           'region_northwest', 'region_southeast', 'region_southwest',
           'bmi_category_normal', 'bmi_category_overweight', 'bmi_category_obese']
correlations = {
    feature: pearsonr(df_cleaned[feature] , df_cleaned['charges'])[0]
    for feature in selected_features
}

In [ ]:
corr_df = pd.DataFrame(list(correlations.items()),
                       columns=['feature', 'correlation'])
corr_df = corr_df.sort_values(by='correlation', ascending=False)


In [ ]:
corr_df

In [ ]:
from scipy.stats import chi2_contingency


In [ ]:
df_cleaned['charges_category'] = pd.qcut(
    df_cleaned['charges'],
    q=4,
    labels=['low', 'medium_low', 'medium_high', 'high']
)

In [ ]:
categorical_features = [
    'is_female',
    'is_smoker',
    'region_northwest',
    'region_southeast',
    'region_southwest',
    'bmi_category_normal',
    'bmi_category_overweight',
    'bmi_category_obese'
]

In [ ]:
alpha = 0.05   # significance level

results = []

for feature in categorical_features:
    
    # contingency table
    table = pd.crosstab(df_cleaned[feature], df_cleaned['charges_category'])
    
    # chi-square test
    chi2, p, dof, expected = chi2_contingency(table)
    
    # decision rule
    if p < alpha:
        decision = "Reject H0 (Significant relationship)"
    else:
        decision = "Fail to reject H0 (No significant relationship)"
    
    results.append([feature, chi2, p, decision])

In [ ]:
chi_df = pd.DataFrame(
    results,
    columns=['feature', 'chi2_stat', 'p_value', 'decision']
)

In [ ]:
chi_df = chi_df.sort_values(by='p_value')

In [ ]:
chi_df

In [ ]:
final_df = df_cleaned[[
    'age',
    'bmi',
    'children',
    'is_female',
    'is_smoker',
    'charges',
    'region_southeast',
    'bmi_category_obese'
]]


In [ ]:
final_df